# Analyse Graphique — Détection de Bots Twitter
## Phase 1.4 : Construction du graphe d'interactions et features graphiques

Ce notebook exécute le pipeline d'analyse graphique :
- Construction du graphe d'interactions Twitter (nœuds = utilisateurs, arêtes = mentions)
- Calcul de 5 features graphiques (Degree Centrality, PageRank, Clustering Coefficient...)
- Fusion avec les features tabulaires de la Phase 1.3
- Exportation du dataset final enrichi

## 1. Importation des modules et initialisation

In [1]:
import sys
sys.path.append('../src/utils')

from graph_features import GraphFeatureExtractor
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('✓ Modules importés avec succès')

✓ Modules importés avec succès


## 2. Exécution du pipeline graphique complet

In [2]:
# Exécution du pipeline complet d'analyse graphique
extractor = GraphFeatureExtractor(
    raw_path='../bot_detection_data.csv',
    processed_path='../data/processed_features.csv',
    output_path='../data/graph_features.csv'
)
df_final = extractor.process()
print('\n→ Le graphe simulé représente les interactions de mention entre 50 000 utilisateurs. Les 7 mesures de centralité calculées encodent le rôle structurel de chaque compte dans le réseau social.')


🚀 DEBUT DU PIPELINE D'ANALYSE GRAPHIQUE — Phase 1.4
📥 Chargement des données...
✓ Dataset chargé : 50000 lignes × 11 colonnes

🔗 Construction du graphe d'interactions...
✓ Graphe construit :
   Noeuds : 50,000
   Arêtes : 125,688

⚙️  Calcul des 7 features graphiques...
   1/7 Degree Centrality...
   2/7 In-Degree Centrality...
   3/7 Out-Degree Centrality...
   4/7 Closeness Centrality (Eppstein-Wang k=100)...
   5/7 Betweenness Centrality (k=100)...
   6/7 PageRank...
   7/7 Clustering Coefficient...

✓ 7 features graphiques calculées pour 50,000 utilisateurs

🔀 Fusion avec les features tabulaires (Phase 1.3)...
✓ Dataset final : 50,000 lignes × 16 colonnes
   Features tabulaires : 8
   Features graphiques : 7
   Colonne cible       : Bot Label

💾 Sauvegardé dans : ../data/graph_features.csv
   Dimensions : (50000, 16)

📊 Statistiques des features graphiques :
       degree_centrality  in_degree_centrality  out_degree_centrality  closeness_centrality  betweenness_centrality      page

## 3. Inspection du dataset final

In [3]:
# Vérification de la structure du dataset final enrichi
df = pd.read_csv('../data/graph_features.csv')

print(f'Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes')
print(f'\nColonnes disponibles :')
print(df.columns.tolist())
print('\n→ Le dataset final combine les 8 features tabulaires (Phase 1.3) et 7 features graphiques (Phase 1.4), soit 15 features explicatives pour un total de 50 000 observations.')


Dimensions : 50000 lignes × 16 colonnes

Colonnes disponibles :
['followers_to_retweet_ratio', 'retweet_to_mention_ratio', 'account_age_days', 'is_verified', 'tweet_length', 'hashtag_count', 'mentions_count', 'engagement_score', 'Bot Label', 'degree_centrality', 'in_degree_centrality', 'out_degree_centrality', 'closeness_centrality', 'betweenness_centrality', 'pagerank', 'clustering_coefficient']

→ Le dataset final combine les 8 features tabulaires (Phase 1.3) et 7 features graphiques (Phase 1.4), soit 15 features explicatives pour un total de 50 000 observations.


In [4]:
# Aperçu des premières lignes du dataset enrichi
print(df.head().to_string())
print("\n→ On vérifie que les features graphiques ont bien été associées à chaque utilisateur. Les valeurs de centralité varient selon l\'activité du compte dans le réseau.")



   followers_to_retweet_ratio  retweet_to_mention_ratio  account_age_days  is_verified  tweet_length  hashtag_count  mentions_count  engagement_score  Bot Label  degree_centrality  in_degree_centrality  out_degree_centrality  closeness_centrality  betweenness_centrality  pagerank  clustering_coefficient
0                   -0.330667                  1.088775          1.360075     -1.00016      1.236852      -1.462696       -0.885993          1.124392          1            0.00002                   0.0                0.00002                   0.0                     0.0  0.000006                     0.0
1                   -0.125117                 -0.548522         -1.214518      0.99984      0.872584      -0.292661        1.455179         -0.000256          0            0.00010                   0.0                0.00010                   0.0                     0.0  0.000006                     0.0
2                    0.517787                 -0.900541         -0.909668      0.9998

In [5]:
# Statistiques descriptives des 7 features graphiques
graph_cols = ['degree_centrality', 'in_degree_centrality', 'out_degree_centrality',
              'closeness_centrality', 'betweenness_centrality', 'pagerank', 'clustering_coefficient']
print(df[graph_cols].describe().round(6).to_string())
print("\n→ Les distributions sont fortement asymétriques (mean << max), ce qui indique qu\'une minorité de comptes très actifs dominent le graphe. Ce phénomène est typique des réseaux sociaux (loi de puissance) et est souvent corrélé avec un comportement de bot coordiné.")



       degree_centrality  in_degree_centrality  out_degree_centrality  closeness_centrality  betweenness_centrality      pagerank  clustering_coefficient
count       50000.000000          50000.000000           50000.000000               50000.0            50000.000000  50000.000000            50000.000000
mean            0.000101              0.000050               0.000050                   0.0                0.000000      0.000020                0.015517
std             0.000794              0.000794               0.000034                   0.0                0.000005      0.000270                0.077173
min             0.000000              0.000000               0.000000                   0.0                0.000000      0.000006                0.000000
25%             0.000020              0.000000               0.000020                   0.0                0.000000      0.000006                0.000000
50%             0.000060              0.000000               0.000060       

## 4. Visualisation — Distribution des features graphiques par classe

In [6]:
# Visualisation de la distribution des features graphiques par classe (Bot vs Humain)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Distribution des features graphiques : Bots vs Humains', fontsize=14, fontweight='bold')

graph_cols = ['degree_centrality', 'in_degree_centrality', 'out_degree_centrality',
              'closeness_centrality', 'betweenness_centrality', 'pagerank', 'clustering_coefficient']

for idx, col in enumerate(graph_cols):
    ax = axes[idx // 4][idx % 4]
    bots   = df[df['Bot Label'] == 1][col]
    humans = df[df['Bot Label'] == 0][col]
    ax.hist(humans, bins=40, alpha=0.6, color='steelblue', label='Humain')
    ax.hist(bots,   bins=40, alpha=0.6, color='tomato',   label='Bot')
    ax.set_title(col)
    ax.set_xlabel('Valeur')
    ax.set_ylabel('Fréquence')
    ax.legend()

axes[1][3].set_visible(False)
plt.tight_layout()
plt.savefig('../data/graph_features_distribution.png', dpi=150)
plt.show()
print('\n→ Les histogrammes montrent des distributions globalement similaires entre bots et humains pour les centralités, ce qui confirme que ces features apportent un signal complémentaire subtil. Le modèle ML saura les combiner efficacement avec les features tabulaires.')



→ Les histogrammes montrent des distributions globalement similaires entre bots et humains pour les centralités, ce qui confirme que ces features apportent un signal complémentaire subtil. Le modèle ML saura les combiner efficacement avec les features tabulaires.


## 5. Résumé

In [7]:
# Synthèse de la Phase 1.4 — Analyse graphique
print('SYNTHÈSE — Phase 1.4 : Analyse Graphique')
print()
print(f'Dataset enrichi sauvegardé dans : ../data/graph_features.csv')
print(f'Dimensions finales               : {df.shape[0]} lignes × {df.shape[1]} colonnes')
print()
print('Récapitulatif des features :')
print('  Phase 1.3 — Features tabulaires : 8 (ratios, âge, longueur, engagement)')
print('  Phase 1.4 — Features graphiques : 7 (degree, in/out-degree, closeness, betweenness, pagerank, clustering)')
print('  Total                           : 15 features explicatives')
print()
print('Prochaine étape :')
print('  Phase 1.5 — Entraînement et comparaison de 3 modèles ML')
print('             XGBoost | RandomForest | LightGBM')


SYNTHÈSE — Phase 1.4 : Analyse Graphique

Dataset enrichi sauvegardé dans : ../data/graph_features.csv
Dimensions finales               : 50000 lignes × 16 colonnes

Récapitulatif des features :
  Phase 1.3 — Features tabulaires : 8 (ratios, âge, longueur, engagement)
  Phase 1.4 — Features graphiques : 7 (degree, in/out-degree, closeness, betweenness, pagerank, clustering)
  Total                           : 15 features explicatives

Prochaine étape :
  Phase 1.5 — Entraînement et comparaison de 3 modèles ML
             XGBoost | RandomForest | LightGBM
